# Tyro Analysis Toolkit — self-contained attack bench + measurement

**Team Tyro** · FIT5230 Theme 2 **Dark** · target: PhotoGuard · baseline: [IMPRESS](https://github.com/AAAAAAsuka/Impress) (NeurIPS 2023)

---

## Two ways to run this notebook

| | `MODE = "standalone"` | `MODE = "impress"` |
|---|---|---|
| **needs** | a blank Colab T4 and a few face photos | IMPRESS stages 8a–8d already run **in this same runtime** |
| **runtime** | ~15 min end to end | seconds (it only reads files) |
| **protection** | PhotoGuard **encoder attack**, ~30 s/image | PhotoGuard **diffusion attack**, ~77 min/image |
| **purifiers compared** | JPEG · blur · FFT low-pass · IMPRESS-lite | whatever IMPRESS produced |
| **use it for** | fast iteration, M1 submission, trying ideas | the authoritative numbers for M2/M3 |

**Why standalone mode exists.** PhotoGuard has two attacks. The *diffusion attack* is the
77-minute-per-image monster we measured in `HANDOVER.md`. The *encoder attack* — projected
gradient descent against the VAE encoder alone — takes about **30 seconds** and produces a
shield of the same character. Editing is cheap too. So the entire clean → protect → purify
→ edit → measure loop fits in one coffee break, on a runtime with nothing installed.

That matters for marks, not just convenience: M1 asks for *your own* Colab, and M3's Colab
rubric explicitly wants *"a complete, functional workflow from start to finish"* with
*"pre-rendered execution outputs"*. A notebook that runs top-to-bottom on a blank runtime
is far easier to submit than one with a hidden 77-minute prerequisite.

> **This does not replace IMPRESS as our baseline.** IMPRESS stays the paper we cite and
> improve. Standalone mode is the fast loop we develop ideas in; the real IMPRESS run
> supplies the authoritative numbers. Section 1.5 reimplements IMPRESS's *core objective*
> in ~15 lines so that even in standalone mode we are comparing against the baseline's
> actual idea rather than a straw man.

## What gets measured

**The attack-success axis** — each *edited* image is scored against the *edit prompt* with
CLIP. Shield works → the edit disobeys → low score. Our purification works → score climbs
back. One reportable number per purifier:

$$R = \frac{S_{\text{purified}} - S_{\text{protected}}}{S_{\text{clean}} - S_{\text{protected}}}$$

$R = 1$ means we fully restored the editor; $R = 0$ means the shield held.

**The fidelity axis** — how close the purified image still is to the original photo. An
attack that restores editing by turning the face to mush has won nothing.

**The frequency signature** — where in the spectrum the shield lives, which gives us both a
surgical filter design and a detector.

> **Metaphor.** IMPRESS sands the whole table to remove one scratch. Part B finds exactly
> where the scratch is so we can sand only that.


## 0 · Configuration

**Everything is driven by this one cell.** In `impress` mode the folder names are built
from the run parameters, so `PARAMS` must match *exactly* what you passed to stages 8a–8d
(check with `!ls /content/helen_face`). In `standalone` mode `PARAMS` is just a label.

In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors scikit-image pandas matplotlib
print("ok")


In [ ]:
import os, glob, math, json, io
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

# ============================== CONFIGURE ME =================================
MODE = "standalone"          # "standalone"  -> build everything here, no IMPRESS needed
                             # "impress"     -> read folders IMPRESS already wrote

ROOT   = "/content/tyro" if MODE == "standalone" else "/content/helen_face"
PARAMS = "encoder_eps16_steps200" if MODE == "standalone" else "pg_iters40_grad_reps2_diff_steps4"

# The instruction given to the AI editor. CLIP scores are measured against this
# string, so it must describe what the edit was actually asked to do.
EDIT_PROMPT = "a photo of a person wearing a large red hat"

# --- standalone-only knobs ---
SOURCE_DIR = "/content/source_faces"   # put your own images here; upload prompt if empty
N_IMAGES   = 4                          # >= 2 for meaningful std; 4 keeps runtime ~15 min
IMG_SIZE   = 512
PG_EPS     = 16/255                     # PhotoGuard perturbation budget (L-inf, [0,1] scale)
PG_STEPS   = 200                        # PGD steps for the encoder attack
EDIT_STRENGTH = 0.6                     # img2img strength: how much the editor rewrites
# =============================================================================

assert MODE in ("standalone", "impress")

if MODE == "standalone":
    # name -> folder. "clean" and "protected" are inputs; the rest are purifiers.
    VARIANTS = ["clean", "protected", "jpeg", "blur", "fft", "impress_lite"]
    VARIANT_DIRS = {v: f"{ROOT}/{v}" for v in VARIANTS}
    EDIT_DIRS    = {v: f"{ROOT}/{v}_edit" for v in VARIANTS}
    PURIFIERS    = ["jpeg", "blur", "fft", "impress_lite"]
else:
    VARIANTS = ["clean", "protected", "impress"]
    VARIANT_DIRS = {"clean":     f"{ROOT}/clean",
                    "protected": f"{ROOT}/adv_{PARAMS}",
                    "impress":   f"{ROOT}/pur_{PARAMS}"}
    EDIT_DIRS    = {"clean":     f"{ROOT}/clean_diff",
                    "protected": f"{ROOT}/adv_diff_{PARAMS}",
                    "impress":   f"{ROOT}/pur_diff_{PARAMS}"}
    PURIFIERS    = ["impress"]

for d in list(VARIANT_DIRS.values()) + list(EDIT_DIRS.values()):
    os.makedirs(d, exist_ok=True)

# Palette: Okabe-Ito subset, validated colourblind-safe
# (worst adjacent pair deutan dE 11.0 / normal dE 25.8; all >= 3:1 contrast on white).
PAL = {"clean": "#0072B2", "protected": "#D55E00", "impress": "#009E73",
       "impress_lite": "#009E73", "jpeg": "#7a7a7a", "blur": "#4b4b4b", "fft": "#CC79A7"}

print(f"MODE = {MODE}")
print(f"ROOT = {ROOT}")
print("variants:", ", ".join(VARIANTS))


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("!! No GPU. Runtime > Change runtime type > T4 GPU.")
    print("!! standalone mode needs a GPU; impress mode (analysis only) will still work.")


## 0.5 · Shared helpers

Image IO, the tensor conversions the VAE expects (it works in `[-1, 1]`, not `[0, 1]` —
getting this wrong is the classic silent bug), and the two DSP functions Part B needs.
They are defined here because standalone mode's FFT purifier uses them too.

In [ ]:
def to_t(img, size=None):
    """PIL -> (1,3,H,W) tensor in [-1,1], which is what the SD VAE expects."""
    if size:
        img = img.resize((size, size), Image.BICUBIC)
    a = np.asarray(img.convert("RGB")).astype(np.float32) / 255.0
    t = torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0)
    return (t * 2 - 1).to(DEVICE)

def to_img(t):
    """(1,3,H,W) tensor in [-1,1] -> PIL."""
    a = ((t.detach().float().clamp(-1, 1) + 1) / 2).squeeze(0).permute(1, 2, 0).cpu().numpy()
    return Image.fromarray((a * 255).round().astype(np.uint8))

def list_images(folder):
    return sorted(p for p in glob.glob(os.path.join(folder, "*.*"))
                  if p.lower().endswith((".png", ".jpg", ".jpeg")))

def load_matched(keys, dirs):
    """Images whose basenames appear in EVERY requested folder, so every metric
    is computed on the same face across conditions."""
    sets = [{os.path.basename(p) for p in list_images(dirs[k])} for k in keys]
    common = sorted(set.intersection(*sets)) if sets else []
    out = {k: [Image.open(os.path.join(dirs[k], n)).convert("RGB") for n in common]
           for k in keys}
    return common, out

eps_num = 1e-12


In [ ]:
def radial_spectrum(img, nbins=180):
    """Direction-averaged FFT magnitude vs normalised spatial frequency (0=DC, 1=Nyquist)."""
    g = np.asarray(img.convert("L"), dtype=np.float64) / 255.0
    g = g - g.mean()                                   # drop DC so it can't dominate
    mag = np.abs(np.fft.fftshift(np.fft.fft2(g)))

    h, w = mag.shape
    y, x = np.indices((h, w))
    r = np.sqrt((y - h/2.0)**2 + (x - w/2.0)**2) / (min(h, w) / 2.0)

    keep = r <= 1.0
    bins = np.linspace(0, 1.0, nbins + 1)
    idx  = np.clip(np.digitize(r[keep], bins) - 1, 0, nbins - 1)
    tot  = np.bincount(idx, weights=mag[keep], minlength=nbins)
    cnt  = np.bincount(idx, minlength=nbins)
    return 0.5 * (bins[:-1] + bins[1:]), tot / np.maximum(cnt, 1)

def butterworth_lowpass(img, f_c, order=4):
    """Attenuate frequencies above f_c. Butterworth, not a hard cut: a brick-wall
    filter causes ringing (Gibbs), which would cost fidelity for no benefit."""
    a = np.asarray(img.convert("RGB"), dtype=np.float64) / 255.0
    h, w, _ = a.shape
    y, x = np.indices((h, w))
    r = np.sqrt((y - h/2.0)**2 + (x - w/2.0)**2) / (min(h, w) / 2.0)
    H = 1.0 / (1.0 + (r / max(f_c, 1e-6))**(2 * order))

    out = np.zeros_like(a)
    for c in range(3):
        F = np.fft.fftshift(np.fft.fft2(a[..., c]))
        out[..., c] = np.real(np.fft.ifft2(np.fft.ifftshift(F * H)))
    return Image.fromarray(np.clip(out * 255, 0, 255).astype(np.uint8))

print("helpers ready")


---

# Section 1 · Build the data *(standalone mode only)*

Every cell in this section is a no-op when `MODE = "impress"`. Skip straight to Part A in
that case.

## 1.1 · Source images

Put a few face photos in `SOURCE_DIR`, or let the cell open an upload prompt. Anything
works — the pipeline resizes to `IMG_SIZE`. Faces are conventional here because PhotoGuard's
own evaluation uses them, but the method is not face-specific.

If you want the same Helen faces IMPRESS uses:

```python
!pip install -q gdown && gdown 16xISe7M_DlSqM2Zf2lWI4JJXcdsDEPsl
!unzip -q helen_face_dataset.zip -d /content/ && mkdir -p /content/source_faces
!cp /content/helen_face/clean/*.png /content/source_faces/ 2>/dev/null | head
```

In [ ]:
if MODE == "standalone":
    os.makedirs(SOURCE_DIR, exist_ok=True)
    srcs = list_images(SOURCE_DIR)

    if not srcs:
        try:
            from google.colab import files
            print(f"No images in {SOURCE_DIR}. Upload {N_IMAGES}+ photos (any size):")
            up = files.upload()
            for name, data in up.items():
                if name.lower().endswith((".png", ".jpg", ".jpeg")):
                    with open(os.path.join(SOURCE_DIR, name), "wb") as f:
                        f.write(data)
            srcs = list_images(SOURCE_DIR)
        except ImportError:
            raise SystemExit(f"Not in Colab. Put images in {SOURCE_DIR} manually.")

    srcs = srcs[:N_IMAGES]
    assert srcs, f"Still no images in {SOURCE_DIR}"
    if len(srcs) < 2:
        print("!! Only 1 image. Every std below will be undefined. Use >= 2, ideally >= 5.")

    NAMES = []
    for p in srcs:
        n = os.path.splitext(os.path.basename(p))[0] + ".png"
        Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC) \
             .save(os.path.join(VARIANT_DIRS["clean"], n))
        NAMES.append(n)
    print(f"prepared {len(NAMES)} clean images at {IMG_SIZE}x{IMG_SIZE}:", NAMES)
else:
    print("impress mode - skipping (folders already exist)")


## 1.2 · PhotoGuard's encoder attack

**The idea.** Stable Diffusion never edits your photo directly. It first squeezes the image
through a VAE encoder into a small latent code, edits *that*, and decodes back. PhotoGuard's
encoder attack finds a tiny perturbation that makes the encoder produce the **wrong latent
code** — specifically, it drags the latent toward that of a flat grey image. Downstream, the
editor is working on a photo of nothing.

$$\min_{\|\delta\|_\infty \le \epsilon} \; \big\| E(x + \delta) - E(x_{\text{target}}) \big\|_2^2$$

Solved with projected gradient descent: step along the sign of the gradient, then clip back
into the ε-ball so the perturbation stays invisible.

> **Metaphor.** The VAE encoder is a translator who summarises your photo in one sentence
> before the editor ever sees it. PhotoGuard doesn't touch the photo the editor works on —
> it bribes the translator to say "a grey rectangle" no matter what you hand over.

`eps = 16/255` is the standard budget: each pixel may move by at most about 6% of full
range. You will not see it. The encoder will.

In [ ]:
if MODE == "standalone":
    import torch.nn.functional as Fn
    from diffusers import AutoencoderKL

    MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    # fp32 for the attack: fp16 gradients underflow and the PGD silently does nothing.
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae",
                                        torch_dtype=torch.float32).to(DEVICE).eval()
    for p in vae.parameters():
        p.requires_grad_(False)
    print("VAE loaded (fp32)")

def photoguard_encoder_attack(img, eps=PG_EPS, steps=PG_STEPS, alpha=None, verbose=False):
    """PGD against the VAE encoder. Returns the protected PIL image."""
    alpha = alpha or (eps / steps * 8)
    x = to_t(img, IMG_SIZE)
    eps_t, alpha_t = eps * 2, alpha * 2          # [0,1] budget -> [-1,1] tensor scale

    with torch.no_grad():
        target = torch.zeros_like(x)             # flat grey in [-1,1]
        z_tgt  = vae.encode(target).latent_dist.mean

    delta = torch.zeros_like(x, requires_grad=True)
    for i in range(steps):
        z = vae.encode(x + delta).latent_dist.mean
        loss = Fn.mse_loss(z, z_tgt)
        grad, = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            delta -= alpha_t * grad.sign()                     # descend: toward the target
            delta.clamp_(-eps_t, eps_t)                        # project into the eps-ball
            delta.copy_((x + delta).clamp(-1, 1) - x)          # keep pixels valid
        if verbose and (i + 1) % 50 == 0:
            print(f"    step {i+1}/{steps}  latent MSE {loss.item():.5f}")
    return to_img(x + delta)

print("attack defined")


In [ ]:
if MODE == "standalone":
    import time
    t0 = time.time()
    for i, n in enumerate(NAMES):
        clean = Image.open(os.path.join(VARIANT_DIRS["clean"], n))
        print(f"  [{i+1}/{len(NAMES)}] protecting {n} ...")
        photoguard_encoder_attack(clean, verbose=(i == 0)) \
            .save(os.path.join(VARIANT_DIRS["protected"], n))
    dt = time.time() - t0
    print(f"\ndone in {dt:.0f}s ({dt/len(NAMES):.0f}s per image)")
    print("compare: PhotoGuard's DIFFUSION attack costs ~77 min per image on this GPU.")


In [ ]:
if MODE == "standalone":
    import matplotlib.pyplot as plt
    n = NAMES[0]
    a = np.asarray(Image.open(os.path.join(VARIANT_DIRS["clean"], n)), dtype=np.float64)
    b = np.asarray(Image.open(os.path.join(VARIANT_DIRS["protected"], n)), dtype=np.float64)
    d = b - a

    fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
    ax[0].imshow(a.astype(np.uint8)); ax[0].set_title("clean", fontsize=11)
    ax[1].imshow(b.astype(np.uint8)); ax[1].set_title("PhotoGuard-protected", fontsize=11)
    amp = np.clip(d * 10 + 128, 0, 255).astype(np.uint8)
    ax[2].imshow(amp); ax[2].set_title("the shield, amplified 10x", fontsize=11)
    for a_ in ax: a_.axis("off")
    fig.suptitle(f"max pixel change: {np.abs(d).max():.0f}/255   "
                 f"mean |change|: {np.abs(d).mean():.2f}/255", fontsize=11)
    fig.tight_layout(); fig.savefig("tyro_shield.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Middle image should look identical to the left one. That is the whole point.")


## 1.5 · The purifiers, including a faithful IMPRESS-lite

Four attacks on the shield, cheapest first:

| purifier | idea | cost |
|---|---|---|
| **JPEG q65** | lossy compression discards exactly the fine detail the shield hides in | ~0 s |
| **blur** | Gaussian smoothing, the blunt instrument | ~0 s |
| **FFT low-pass** | Butterworth cut at the band we identify in Part B — *surgical* | ~0 s |
| **IMPRESS-lite** | the baseline's actual objective, reimplemented | ~60 s/image |

**IMPRESS's core idea, honestly reimplemented.** A clean photo survives a round-trip
through the VAE more or less intact. A protected photo does not — that inconsistency *is*
the shield. So IMPRESS optimises the image to be simultaneously **self-consistent under the
round-trip** and **close to the protected input it started from**:

$$\min_{x'} \; \big\| D(E(x')) - x' \big\|_2^2 \;+\; \lambda \big\| x' - x_{\text{prot}} \big\|_2^2$$

The first term erases the shield; the second stops the optimiser from wandering off into an
unrelated image. `LAMBDA` is the dial between them.

> **Metaphor.** A forged signature looks wrong when you trace over it. IMPRESS keeps
> nudging the signature until tracing it reproduces it exactly — at which point whatever
> made it a forgery is gone.

This is a *lite* version — one fixed λ, no scheduling, fewer steps than the paper. It is
here so standalone mode compares against the baseline's real idea rather than a straw man.
Authoritative IMPRESS numbers still come from the real run.

In [ ]:
LAMBDA     = 0.1     # fidelity anchor: higher = stay closer to the protected input
PUR_STEPS  = 150
PUR_LR     = 0.01

def impress_lite(img, steps=PUR_STEPS, lam=LAMBDA, lr=PUR_LR, verbose=False):
    """Optimise for VAE round-trip self-consistency while staying near the input."""
    x_p = to_t(img, IMG_SIZE)
    x   = x_p.clone().requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for i in range(steps):
        rec  = vae.decode(vae.encode(x).latent_dist.mean).sample
        loss = Fn.mse_loss(rec, x) + lam * Fn.mse_loss(x, x_p)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            x.clamp_(-1, 1)
        if verbose and (i + 1) % 50 == 0:
            print(f"    step {i+1}/{steps}  loss {loss.item():.5f}")
    return to_img(x)

def jpeg_purify(img, quality=65):
    buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=quality)
    buf.seek(0); return Image.open(buf).convert("RGB")

def blur_purify(img, radius=1.2):
    return img.filter(ImageFilter.GaussianBlur(radius))

def fft_purify(img, f_c=0.45, order=4):
    return butterworth_lowpass(img, f_c, order)

print("purifiers defined")


In [ ]:
if MODE == "standalone":
    # f_c is provisional here; Part B derives the right value from the data and we
    # re-run this purifier with it in section B3.
    FC_PROVISIONAL = 0.45

    for i, n in enumerate(NAMES):
        prot = Image.open(os.path.join(VARIANT_DIRS["protected"], n))
        jpeg_purify(prot).save(os.path.join(VARIANT_DIRS["jpeg"], n))
        blur_purify(prot).save(os.path.join(VARIANT_DIRS["blur"], n))
        fft_purify(prot, FC_PROVISIONAL).save(os.path.join(VARIANT_DIRS["fft"], n))
        print(f"  [{i+1}/{len(NAMES)}] cheap purifiers done for {n}; running IMPRESS-lite ...")
        try:
            impress_lite(prot, verbose=(i == 0)).save(
                os.path.join(VARIANT_DIRS["impress_lite"], n))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("    OOM at full size - retrying IMPRESS-lite at half resolution")
            small = prot.resize((IMG_SIZE//2, IMG_SIZE//2), Image.BICUBIC)
            impress_lite(small).resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC).save(
                os.path.join(VARIANT_DIRS["impress_lite"], n))
    print("\nall purifiers done")

    del vae
    torch.cuda.empty_cache()      # free fp32 VAE before loading the editor
    print("VAE released")


## 1.7 · Run the AI editor on every variant

This is the step the whole project is about: hand each image to an AI editor with the same
instruction and the same seed, and see who obeys.

We use SD 1.5 img2img, which shares the exact VAE we just attacked — so the shield
transfers, and we avoid a second multi-gigabyte download on a free runtime. To use
InstructPix2Pix instead (closer to PhotoGuard's own demo), swap in
`StableDiffusionInstructPix2PixPipeline` with `"timbrooks/instruct-pix2pix"` and phrase
`EDIT_PROMPT` as an instruction.

**Fixed seed for every image and every variant.** The only thing that differs across
conditions is the input pixels. Without this the comparison proves nothing.

In [ ]:
if MODE == "standalone":
    from diffusers import StableDiffusionImg2ImgPipeline

    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)

    SEED = 5230
    def edit(img):
        g = torch.Generator(device=DEVICE).manual_seed(SEED)
        return pipe(prompt=EDIT_PROMPT, image=img.resize((IMG_SIZE, IMG_SIZE)),
                    strength=EDIT_STRENGTH, guidance_scale=7.5,
                    num_inference_steps=30, generator=g).images[0]

    for v in VARIANTS:
        for n in NAMES:
            src = os.path.join(VARIANT_DIRS[v], n)
            if not os.path.exists(src):
                continue
            edit(Image.open(src).convert("RGB")).save(os.path.join(EDIT_DIRS[v], n))
        print(f"  edited: {v}")
    print("\nall edits done")


In [ ]:
if MODE == "standalone":
    show = [v for v in VARIANTS if os.path.exists(os.path.join(EDIT_DIRS[v], NAMES[0]))]
    fig, axes = plt.subplots(2, len(show), figsize=(2.7*len(show), 5.8))
    for j, v in enumerate(show):
        axes[0, j].imshow(Image.open(os.path.join(VARIANT_DIRS[v], NAMES[0])))
        axes[0, j].set_title(v, fontsize=10, color=PAL.get(v, "#333"))
        axes[1, j].imshow(Image.open(os.path.join(EDIT_DIRS[v],  NAMES[0])))
        for r in (0, 1): axes[r, j].axis("off")
    axes[0, 0].set_ylabel("input");  axes[1, 0].set_ylabel("edited")
    fig.suptitle(f'top: what we fed the editor   |   bottom: what it produced\n"{EDIT_PROMPT}"',
                 fontsize=11)
    fig.tight_layout(); fig.savefig("tyro_story_panel.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Look at 'protected': if the shield worked, that edit is visibly damaged.")
    print("Look at the purifiers: if our attack worked, theirs look like 'clean' again.")


---

# Part A · The attack-success axis

## A1 · An independent judge

We score edited images with `openai/clip-vit-base-patch32` — deliberately **not** the
ViT-L/14 encoder inside Stable Diffusion. Scoring with the model that generated the image
would flatter our results.

> **Metaphor.** You do not let the chef grade the dish. You bring in a diner who has never
> been in the kitchen.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

EVAL_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(EVAL_ID).to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained(EVAL_ID)

@torch.no_grad()
def clip_scores(images, text, batch=16):
    out = []
    for i in range(0, len(images), batch):
        inp = clip_proc(text=[text], images=images[i:i+batch], return_tensors="pt",
                        padding=True, truncation=True).to(DEVICE)
        ie = clip_model.get_image_features(pixel_values=inp["pixel_values"])
        te = clip_model.get_text_features(input_ids=inp["input_ids"],
                                          attention_mask=inp["attention_mask"])
        ie = ie / ie.norm(dim=-1, keepdim=True)
        te = te / te.norm(dim=-1, keepdim=True)
        out.extend((ie @ te.T).squeeze(-1).cpu().tolist())
    return out

print("independent judge loaded:", EVAL_ID)


In [ ]:
avail = [v for v in VARIANTS if list_images(EDIT_DIRS[v])]
print("edited folders found:", avail)
missing = [v for v in VARIANTS if v not in avail]
if missing:
    print("missing:", missing, "-> in impress mode this is almost always a PARAMS mismatch;")
    print("   run  !ls", ROOT)

names, edits = load_matched(avail, EDIT_DIRS)
assert names, "No common filenames across edited folders."
print(f"\n{len(names)} images matched across all edited folders\n")

df = pd.DataFrame({v: clip_scores(edits[v], EDIT_PROMPT) for v in avail}, index=names)
df.index.name = "image"
print(f'CLIP( edited image , "{EDIT_PROMPT}" )   higher = the editor obeyed\n')
print(df.round(4).to_string())
print()
print(df.describe().loc[["mean", "std"]].round(4).to_string())


## A2 · Protection efficacy and restoration rate

Read the guard rail carefully. If the shield never blocked the edit in the first place,
protection efficacy `P` sits in the noise and `R` becomes a ratio of two nearly-zero
numbers — it will print a confident-looking value that means **nothing**. This is exactly
what happened in Trial #1 with light settings. The cell refuses to let it pass silently.

In [ ]:
m = df.mean()
m_clean, m_prot = m["clean"], m["protected"]
P = m_clean - m_prot
noise = df.std().mean()

print(f"  S_clean     = {m_clean:.4f}   (editor on an unprotected photo)")
print(f"  S_protected = {m_prot:.4f}   (editor on a shielded photo)")
print(f"  Protection efficacy  P = {P:+.4f}   (within-condition spread {noise:.4f})\n")

if len(df) < 2:
    print("  !! Only 1 image - every std is undefined. Use >= 5 before reporting.\n")

if P <= max(noise, 0.005):
    print("  !! PROTECTION EFFICACY IS BELOW THE NOISE FLOOR.")
    print("  !! The shield is not blocking the edit, so there is nothing to restore.")
    print("  !! R is NOT INTERPRETABLE. Do not report it. Strengthen the shield first:")
    print("  !!   standalone -> raise PG_STEPS / PG_EPS")
    print("  !!   impress    -> raise --pg_iters toward 200, --pg_grad_reps toward 10")
    R = {v: float("nan") for v in PURIFIERS if v in avail}
else:
    R = {v: (m[v] - m_prot) / P for v in PURIFIERS if v in avail}
    print("  RESTORATION RATE  R = (S_purified - S_protected) / P")
    print("  R = 1.0 -> editor fully restored.   R = 0 -> shield held.\n")
    for v, r in sorted(R.items(), key=lambda kv: -kv[1]):
        verdict = ("attack succeeded" if r > 0.8 else
                   "partial strip"    if r > 0.4 else "shield held")
        print(f"    {v:14s} R = {r:+.3f}   {verdict}")


## A3 · Fidelity — what did the attack cost the image?

The second axis. An attack that restores editing by wrecking the photo has won nothing.

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

def arr(im, size=None):
    if size and im.size != size:
        im = im.resize(size, Image.BICUBIC)
    return np.asarray(im.convert("RGB")).astype(np.float64)

in_avail = [v for v in VARIANTS if list_images(VARIANT_DIRS[v])]
names_in, ins = load_matched(in_avail, VARIANT_DIRS)

rows = []
for i, n in enumerate(names_in):
    ref, sz = arr(ins["clean"][i]), ins["clean"][i].size
    row = {"image": n}
    for v in in_avail:
        if v == "clean":
            continue
        a = arr(ins[v][i], sz)
        row[f"ssim_{v}"] = ssim(ref, a, channel_axis=2, data_range=255)
        row[f"psnr_{v}"] = psnr(ref, a, data_range=255)
    rows.append(row)

fid = pd.DataFrame(rows).set_index("image")
print("Similarity to the ORIGINAL clean photo (SSIM 1.0 = identical)\n")
print(fid[[c for c in fid.columns if c.startswith("ssim")]].round(4).to_string())
print()
print(fid.mean().round(4).to_string())


## A4 · The money chart — fidelity vs attack success

**This is the figure for the M2 post and the M3 deck.** Each purifier is one point. The
ideal sits **top right**: restores the editor completely, barely touches the image.

One y-axis, deliberately. Fidelity and attack success are different quantities on different
scales; twin axes would be the classic charting mistake. They get a scatter.

In [ ]:
points = [("clean (upper bound)", 1.0, m_clean, "clean"),
          ("PhotoGuard-protected", fid.get("ssim_protected", pd.Series([np.nan])).mean(),
           m_prot, "protected")]
for v in PURIFIERS:
    if v in avail and f"ssim_{v}" in fid:
        points.append((v, fid[f"ssim_{v}"].mean(), m[v], v))

fig, ax = plt.subplots(figsize=(8, 5.8))
for label, x, y, key in points:
    ax.scatter(x, y, s=150, color=PAL.get(key, "#666"), zorder=3,
               edgecolor="white", linewidth=2)          # 2px surface ring
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 15),
                ha="center", fontsize=9, color="#333333")

ax.axhline(m_clean, color=PAL["clean"], linestyle=":", linewidth=1.2, zorder=1)
ax.axhline(m_prot,  color=PAL["protected"], linestyle=":", linewidth=1.2, zorder=1)
ax.set_xlabel("Image fidelity  —  SSIM vs the original photo  →")
ax.set_ylabel("Attack success  —  CLIP(edit, prompt)  →")
ax.set_title("Tyro purifiers: how much image do we destroy to restore the editor?",
             fontsize=12, pad=12)
ax.grid(color="#ececec", linewidth=0.8); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.annotate("ideal", xy=(0.985, 0.985), xycoords="axes fraction", ha="right", va="top",
            fontsize=9, color="#999999", style="italic")
fig.tight_layout(); fig.savefig("tyro_fidelity_vs_success.png", dpi=150, bbox_inches="tight")
plt.show()


---

# Part B · The frequency signature

A 2-D Fourier transform re-describes an image as *how much energy sits at each spatial
frequency*. Low frequencies are broad shapes; high frequencies are fine detail — eyelashes,
pores, and adversarial noise. We collapse the 2-D spectrum to one readable curve per image
by averaging over all directions at each radius.

> **Metaphor.** A photo is a chord. The FFT is the sheet music showing which notes are in it
> and how loud. PhotoGuard adds a note you cannot consciously hear — but it is right there
> on the page.

In [ ]:
spec_keys = [v for v in ["clean", "protected"] + PURIFIERS if v in in_avail]
spec = {k: [] for k in spec_keys}
for i in range(len(names_in)):
    sz = ins["clean"][i].size
    for k in spec_keys:
        im = ins[k][i]
        if im.size != sz:
            im = im.resize(sz, Image.BICUBIC)
        f, p = radial_spectrum(im)
        spec[k].append(p)
freq = f
logs = {k: np.log10(np.array(v) + eps_num) for k, v in spec.items()}

# Paired per-image differences, then average: each face is its own control.
d = {k: (logs[k] - logs["clean"]).mean(0) for k in spec_keys if k != "clean"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 8.5), sharex=True,
                               gridspec_kw={"height_ratios": [1.2, 1]})
for k in spec_keys:
    ax1.plot(freq, logs[k].mean(0), color=PAL.get(k, "#666"), linewidth=2, label=k)
ax1.set_ylabel("log10 mean FFT magnitude")
ax1.set_title(f"Radial power spectrum, averaged over {len(names_in)} images", fontsize=12)
ax1.legend(frameon=False, fontsize=9, ncol=2)
ax1.grid(color="#ececec", linewidth=0.8); ax1.set_axisbelow(True)
ax1.spines[["top", "right"]].set_visible(False)

ax2.axhline(0, color="#bbbbbb", linewidth=1)
for k, dv in d.items():
    ax2.plot(freq, dv, color=PAL.get(k, "#666"), linewidth=2, label=f"{k} - clean")
ax2.set_xlabel("normalised spatial frequency   (0 = broad shapes,  1.0 = finest detail)")
ax2.set_ylabel("log10 difference")
ax2.set_title("Where the shield lives — positive = energy PhotoGuard injected", fontsize=11)
ax2.legend(frameon=False, fontsize=9, ncol=2)
ax2.grid(color="#ececec", linewidth=0.8); ax2.set_axisbelow(True)
ax2.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("tyro_fft_signature.png", dpi=150, bbox_inches="tight")
plt.show()


## B2 · Where is the band, and where should we cut?

We take the frequency at which the injected energy first passes half its peak as the
**onset** of the perturbation band. A filter placed there removes most of the shield and
leaves everything below — i.e. most of the actual face — alone.

That is the filter spec. Read the other way, it is also the **detector**.

In [ ]:
d_prot = d["protected"]
pi     = int(np.argmax(d_prot))
half   = 0.5 * d_prot[pi]
oi     = int(np.argmax(d_prot >= half)) if (d_prot >= half).any() else pi
onset_f = float(freq[oi])
band    = freq >= onset_f

print(f"  peak injected energy at f = {freq[pi]:.3f}  ({d_prot[pi]:+.4f} log10)")
print(f"  band onset (half-peak)    f = {onset_f:.3f}")
print(f"  --> SUGGESTED LOW-PASS CUTOFF  f_c = {onset_f:.3f}\n")

e_prot = d_prot[band].mean()
print(f"  in-band energy, protected vs clean: {e_prot:+.4f} log10\n")
for k in PURIFIERS:
    if k not in d:
        continue
    res = d[k][band].mean() / max(abs(e_prot), 1e-9)
    print(f"  {k:14s} in-band {d[k][band].mean():+.4f}  ->  "
          f"{100*(1-abs(res)):5.1f}% of the shield neutralised", end="")
    # Residue can go NEGATIVE: the purifier removed more than PhotoGuard added and ate
    # into the image's own detail. An unnaturally SMOOTH image is a fingerprint too.
    if   res >  0.02: print("   (UNDER-filtered: noise left behind)")
    elif res < -0.02: print("   (OVER-filtered: real detail removed - also detectable)")
    else:             print("   (well matched)")
print("\n  These residues set the difficulty of our Tier-3 detection challenge. B4 scores it.")


## B3 · Tune the surgical filter

Now that the data has told us where the band is, sweep `f_c` around the suggested cutoff
and watch the fidelity/strip trade-off directly. Pick a value, then re-run the FFT purifier
and the editor with it to place a properly-tuned point on the Part A chart.

In [ ]:
sweep = [round(onset_f + s, 3) for s in (-0.10, -0.05, 0.0, 0.05, 0.10)]
sweep = [c for c in sweep if 0.05 < c < 0.95]

rows = []
for fc in sweep:
    ss, be = [], []
    for i, n in enumerate(names_in):
        ref, sz = arr(ins["clean"][i]), ins["clean"][i].size
        out = butterworth_lowpass(ins["protected"][i], fc, order=4)
        ss.append(ssim(ref, arr(out, sz), channel_axis=2, data_range=255))
        fq, p = radial_spectrum(out)
        be.append(np.log10(p[fq >= onset_f] + eps_num).mean())
    rows.append({"f_c": fc, "ssim_vs_clean": np.mean(ss), "in_band_energy": np.mean(be)})

sw = pd.DataFrame(rows)
clean_band = np.mean([np.log10(radial_spectrum(im)[1][freq >= onset_f] + eps_num).mean()
                      for im in ins["clean"]])
sw["gap_vs_clean"] = sw["in_band_energy"] - clean_band
print("Lower |gap_vs_clean| = harder to detect. Higher ssim = less damage.\n")
print(sw.round(4).to_string(index=False))
best = sw.iloc[sw["gap_vs_clean"].abs().argmin()]
print(f"\n--> least-detectable cutoff: f_c = {best['f_c']:.3f} "
      f"(SSIM {best['ssim_vs_clean']:.4f}, gap {best['gap_vs_clean']:+.4f})")
print("--> set FC_PROVISIONAL to this, re-run 1.6 and 1.7, and the FFT point on the")
print("    Part A chart becomes the tuned one. Fidelity alone is not the goal:")
print("    a filter that is faithful but does not restore the editor has won nothing.")


## B4 · The detection answer key — private, do not post

Our Tier-3 challenge asks other teams to sort clean / protected / purified images. Before we
publish it we should know how hard it actually is. This is a deliberately *simple* detector
— one number per image, one threshold — so it is the **floor** of what an opposing team
could manage. If this already separates the classes cleanly, our challenge is too easy and
our attack is too visible.

> **Adversarial-gameplay note for the M4 log:** publishing a challenge whose answer you have
> already computed privately is *keen observation* in the rubric's sense. We choose the
> difficulty rather than discovering it.

In [ ]:
def band_energy(img, f_lo):
    fq, p = radial_spectrum(img)
    return float(np.log10(p[fq >= f_lo] + eps_num).mean())

feat = {k: np.array([band_energy(im, onset_f) for im in ins[k]]) for k in in_avail}
print(f"Band energy above f = {onset_f:.3f}\n")
for k, v in feat.items():
    print(f"  {k:14s} {v.mean():+.4f} +/- {v.std():.4f}   n={len(v)}")

allv = np.concatenate([feat["clean"], feat["protected"]])
lab  = np.concatenate([np.zeros(len(feat["clean"])), np.ones(len(feat["protected"]))])
acc  = max(max(((allv > t) == lab).mean(), ((allv <= t) == lab).mean())
           for t in np.linspace(allv.min(), allv.max(), 400))
print(f"\n  1-D threshold detector, clean vs protected: {100*acc:.1f}% accuracy")

print("\n  OUR EXPOSURE - how far each purified set sits from clean:")
pooled = np.concatenate([feat["clean"]] + [feat[k] for k in PURIFIERS if k in feat])
sd = max(pooled.std(), 1e-9)
for k in PURIFIERS:
    if k not in feat:
        continue
    sep = abs(feat[k].mean() - feat["clean"].mean()) / sd
    tag = "EASILY DETECTED" if sep > 1.5 else "hard to detect - good"
    print(f"    {k:14s} {sep:.2f} sigma from clean   <- {tag}")
print("\n  Anything above ~1.5 sigma will lose us the Tier-3 challenge. Retune before posting.")

fig, ax = plt.subplots(figsize=(8, 4.2))
for i, k in enumerate(in_avail):
    ax.scatter(feat[k], np.full(len(feat[k]), i), s=110, color=PAL.get(k, "#666"),
               edgecolor="white", linewidth=2, zorder=3)
ax.set_yticks(range(len(in_avail))); ax.set_yticklabels(in_avail)
ax.set_xlabel(f"mean log10 band energy above f = {onset_f:.3f}")
ax.set_title("Detection answer key: does one number separate the classes?", fontsize=11)
ax.grid(axis="x", color="#ececec", linewidth=0.8); ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout(); fig.savefig("tyro_detection_key.png", dpi=150, bbox_inches="tight")
plt.show()


---

## Save everything — Colab is ephemeral

```python
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/FIT5230/figures"
!cp tyro_*.png "/content/drive/MyDrive/FIT5230/figures/"
!cp -r /content/tyro "/content/drive/MyDrive/FIT5230/"     # standalone outputs
```

| file | goes where |
|---|---|
| `tyro_fidelity_vs_success.png` | M2 post, M3 slide 2 — **the headline figure** |
| `tyro_story_panel.png` | M1 Ed post — the clean/protected/purified story |
| `tyro_shield.png` | M1/M3 — shows the shield is invisible |
| `tyro_fft_signature.png` | M3, Echo's individual strategy segment |
| `tyro_detection_key.png` | **private** — sets Tier-3 difficulty, do not post |

## Record in `Echo-M4-Strategy-Log.md`

- **Did:** built a self-contained attack bench — PhotoGuard encoder attack, four purifiers, independent-CLIP success metric, FFT band analysis.
- **Why:** the IMPRESS pipeline costs ~77 min/image, too slow to iterate on. The encoder attack gives the same shape of shield in ~30 s, so ideas can be tested in minutes and confirmed on the real pipeline later.
- **Learned:** shield lives above f ≈ `<onset_f>`; `<best purifier>` gave the highest restoration rate at `<R>`.
- **Interaction / edge:** we know our own detectability before any Light team tests it, so we set the challenge difficulty rather than discovering it.

## Limitations — state these, do not hide them

- **The encoder attack is not the diffusion attack.** It is cheaper and weaker. Every standalone number is a *directional* result; M2/M3 headline figures must come from the real IMPRESS run. Say this explicitly in the Ed post — a team that states its own limits is more credible, not less.
- **IMPRESS-lite is a lite reimplementation** — one fixed λ, fewer steps than the paper. Do not report it as "IMPRESS's performance"; report it as "the baseline objective at reduced budget".
- **CLIPScore measures prompt alignment, not image quality.** Keep the visual panel next to the number.
- **SSIM is a weak perceptual proxy.** Add LPIPS (`pip install lpips`) before M3 if time allows.
- **`onset_f` is estimated at one protection strength** and will move when you raise it. Re-derive after any change to `PG_STEPS`/`PG_EPS` or `pg_iters`.
- **Fewer than 5 images makes every standard deviation meaningless.**
